In [13]:
import pandas as pd
df = pd.read_csv('../test/genai_questions_809156896092057937.csv')
df = pd.read_csv('../test/genai_questions_311016922297059474.csv')
print (df.head(10))
questions = df['questions'].to_list()
print (len(questions))
unique_questions = list(set(questions))
questions = unique_questions
print (len(questions))


                                           questions
0  Can you provide an overview of the wind turbin...
1  What are the most common failure modes for win...
2  Which sensors are available for collecting dat...
3  Are there any specific failure modes that are ...
4  Are there any specific components that are mor...
5  How does the wind turbine gearbox operate, and...
6  Are there any external factors that can affect...
7  Are there any existing maintenance schedules o...
8  Are there any specific failure modes that are ...
9  How do the sensor data and other information c...
407
318


In [14]:
from sentence_transformers import SentenceTransformer, util

option1 = 'all-MiniLM-L6-v2'                 # fast
option2 = 'multi-qa-mpnet-base-dot-v1'       # qa
option3 = 'all-mpnet-base-v2'                # best

model = SentenceTransformer(option1)

#Compute embeddings
embeddings = model.encode(questions, convert_to_tensor=True)

#Compute cosine-similarities for each sentence with each other sentence
cosine_scores = util.cos_sim(embeddings, embeddings)

In [11]:

import torch

cosine_scores = cosine_scores * (1 - torch.eye(cosine_scores.size(0)))

# Calculate mean similarity per sentence
mean_similarity_per_sentence = torch.sum(cosine_scores, dim=1) / (cosine_scores.size(1) - 1)

# Calculate the overall mean similarity
overall_mean_similarity = torch.mean(cosine_scores)

print("Mean Similarity per Sentence:", mean_similarity_per_sentence.numpy())
print("Overall Mean Similarity:", overall_mean_similarity.item())


Mean Similarity per Sentence: [0.5252376  0.4685517  0.50695264 0.42816243 0.2798086  0.5473126
 0.50342923 0.478547   0.44131452 0.43719965 0.36283302 0.29545
 0.40083003 0.26812217 0.4736759  0.5384111  0.2692584  0.30108133
 0.39093238 0.21559793 0.52742624 0.49180454 0.29295632 0.5131986
 0.31941018 0.43546215 0.22092734 0.41283163 0.24252053 0.51943654
 0.5271259  0.25479472 0.5130492  0.4313352  0.5388975  0.52167726
 0.5455731  0.52457756 0.34006956 0.5076922  0.44438344 0.52451265
 0.2833116  0.2971096  0.50907886 0.5141418  0.23685737 0.46865314
 0.5274589  0.51577187 0.34957084 0.51527995 0.5114447  0.45860997
 0.4834659  0.456101   0.28837404 0.570779   0.54378074 0.33615658
 0.52902156 0.49410143 0.47704577 0.23820424 0.47055718 0.3950399
 0.51962    0.51775306 0.4860774  0.5235032  0.4597556  0.47924313
 0.36011335 0.47748297 0.4730608  0.39474955 0.31871408 0.38583243
 0.5498572  0.49014187 0.45783216 0.44824702 0.4599532  0.5424971
 0.5055031  0.554616   0.26032776 0.450

In [12]:
from sklearn.cluster import KMeans
import numpy as np

# Assuming embeddings is a NumPy array or a PyTorch tensor
# embeddings = model.encode(questions, convert_to_tensor=True).numpy()

# Specify the number of clusters
num_clusters = 10

# Perform k-means clustering
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
cluster_assignments = kmeans.fit_predict(embeddings)

# Get cluster centers
cluster_centers = kmeans.cluster_centers_

# Calculate the mean distance of each sample to its cluster center
mean_distances = np.zeros_like(cluster_assignments, dtype=np.float32)
for i, cluster_index in enumerate(cluster_assignments):
    mean_distances[i] = np.linalg.norm(embeddings[i] - cluster_centers[cluster_index])

# Calculate the overall mean distance
overall_mean_distance = np.mean(mean_distances)

print("Mean distance of each sample to its cluster center:", mean_distances)
print("Overall mean distance:", overall_mean_distance)


Mean distance of each sample to its cluster center: [0.54322153 0.5224292  0.46419218 0.7121354  0.6966219  0.503464
 0.47560632 0.63293254 0.6783092  0.6488157  0.575047   0.7283015
 0.7204776  0.5863543  0.5737812  0.5065525  0.55827475 0.7069397
 0.66295147 0.63029796 0.42041352 0.5542316  0.6251062  0.5266858
 0.79243106 0.5807156  0.58787864 0.6272419  0.6364565  0.43056366
 0.41080016 0.6683415  0.46062174 0.58669186 0.46211424 0.49358866
 0.38188356 0.51186365 0.6446733  0.44537413 0.70540845 0.46041745
 0.58832425 0.7872088  0.44198686 0.5554334  0.70764464 0.6944969
 0.5053427  0.5266218  0.6386611  0.50323623 0.52048165 0.612034
 0.5446994  0.53230304 0.7594473  0.41222063 0.4390807  0.5624967
 0.4568764  0.67257273 0.54903626 0.5692651  0.570304   0.7487397
 0.5866451  0.54063886 0.6168873  0.47138414 0.62882453 0.65108794
 0.5831898  0.47559166 0.560226   0.69498515 0.5801397  0.6758044
 0.52507335 0.51281047 0.5468108  0.3803298  0.5386081  0.34396863
 0.24721985 0.509258 